# Step 1: Launch SageMaker Processing for Multimodal Data Prep

Uses `ScriptProcessor` with the sklearn container to run `preprocess.py`.

**Input:** Raw JSONL files in S3.  
**Output:** Filtered train/validation/test JSONL splits in S3.

## Configuration

In [ ]:
import boto3
import sagemaker

region = boto3.session.Session().region_name
sess = sagemaker.session.Session()
bucket = sess.default_bucket()
s3_prefix = "autogluon-multimodal"

s3_input = f"s3://{bucket}/{s3_prefix}/raw/"
s3_output = f"s3://{bucket}/{s3_prefix}/processed"
instance_type = "ml.m5.xlarge"

print(f"Region:      {region}")
print(f"S3 input:    {s3_input}")
print(f"S3 output:   {s3_output}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

if not role_arn:
    raise ValueError("No SageMaker IAM role found. Set role_arn manually.")

print(f"Role: {role_arn}")

## Imports and Processing Image

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output

session = Session()

processing_image = image_uris.retrieve("sklearn", region=region, version="1.2-1")
print(f"Processing image: {processing_image}")

## Create ScriptProcessor

In [ ]:
processor = ScriptProcessor(
    image_uri=processing_image,
    role=role_arn,
    command=["python3"],
    instance_type=instance_type,
    instance_count=1,
    sagemaker_session=session,
)

## Run Processing Job

This launches a SageMaker Processing job that runs `preprocess.py` inside the sklearn container.

In [ ]:
processor.run(
    code="preprocess.py",
    inputs=[
        ProcessingInput(
            input_name="input",
            s3_input=ProcessingS3Input(
                s3_uri=s3_input,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"{s3_output}/train/",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="validation",
            s3_output=ProcessingS3Output(
                s3_uri=f"{s3_output}/validation/",
                local_path="/opt/ml/processing/validation",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"{s3_output}/test/",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
    wait=True,
    logs=True,
)

print(f"\nProcessing complete.")
print(f"Train data:      {s3_output}/train/")
print(f"Validation data: {s3_output}/validation/")
print(f"Test data:       {s3_output}/test/")

## Next Steps

Run `1-training/launch_training.ipynb` with these S3 paths.